# Notebook 03c — Shapley Attribution over Journeys (Monte Carlo)

🎯 **Objective**  
Estimate **campaign credit** using **game-theoretic Shapley values** at the *touchpoint level*.  
For each journey, we approximate how much each campaign **marginally increases the model-predicted conversion probability**, then aggregate across journeys to get **campaign-level Shapley credit** and **share** for BI.

---

## 🧩 Inputs

- **Journeys (all)**: `../data/processed/journeys_all.parquet`  
  Columns used:  
  `uid, conversion, conversion_id, conv_time, campaign_seq, click_seq, cost_seq, time_since_last_click_seq, first_campaign_id, last_campaign_id, total_cost, num_clicks`

---

## 🧱 Feature Space (8 engineered features)

We recompute the **same 8 features** for any *subset* of a journey’s touches so the model can score partial paths:

1) `path_length` (capped at 20)  
2) `num_clicks` (capped at 20)  
3) `total_cost = log1p(sum(cost_seq))`  
4) `avg_cost   = log1p(mean(cost_seq))`  
5) `avg_time_gap = log1p(mean(non-negative time_since_last_click_seq))`  
6) `unique_campaigns`  
7) `first_campaign` (global categorical code)  
8) `last_campaign`  (global categorical code)

> **Consistent coding**: first/last campaign IDs are mapped to integer codes using a **single, global category vocabulary** built from *all* journeys to ensure stable encodings across splits and subsets.

---

## 🤖 The Shapley Booster (8-feature XGBoost)

- If missing, we **train a compact booster** on the 8 features (70/15/15 stratified split).  
- Params (key):  
  `objective=binary:logistic, eval_metric=[logloss, auc], eta=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, seed=42`
- **Observed performance (quick run)**: AUC ≈ **0.957** on both validation and test.  
- Saved to: `../artifacts/models/xgb_booster_shapley.json`

> This model provides **v(S)** = predicted conversion probability for any subset **S** of touches in a journey.

---

## 🧮 Method — Monte Carlo Shapley over Touchpoints

For a single journey with ordered touchpoints:

1. Choose the set of indices to consider:
   - If `CLICKS_ONLY=True`, use only indices where `click_seq[i]==1`.  
   - Fallback to **all touches** if no clicks.
2. Draw `N_PERM` random **permutations** of these indices.
3. For each permutation, add touches one-by-one:
   - Compute `v(S)` before adding, and `v(S ∪ {i})` after adding touch *i*.  
   - The **marginal contribution** is `Δ = v(S ∪ {i}) − v(S)`.  
   - Accumulate Δ to that touch’s **campaign_id**.
4. Average contributions across permutations → **per-campaign Shapley credit** for that journey.
5. Repeat for many journeys; **sum** by campaign.

**Sampling controls**
- `CONV_ONLY=True` — sample only converted journeys (consistent with marketing credit views).
- `N_JOURNEYS=5000` — number of journeys to evaluate.
- `N_PERM=50` — permutations per journey (↑ for smoother estimates).
- `CLICKS_ONLY=False` — credit all touches (toggleable).

**Complexity** ~ `O(N_JOURNEYS × N_PERM × avg_path_length)`.

---

## 📊 Run Summary (this notebook)

- Journeys loaded: **6,814,951** (positives: **806,196**)  
- Booster (8-feature) trained and saved (AUC ≈ **0.957**).  
- Shapley aggregation settings:  
  - `CONV_ONLY=True`, `N_JOURNEYS=5000`, `N_PERM=50`, `CLICKS_ONLY=False`.

**Top campaigns by Shapley credit (sample)**

| campaign_id | shapley_credit | share   |
|---:|---:|---:|
| 10341182 | 225.37 | 0.07398 |
| 32368244 | 191.86 | 0.06298 |
| 15184511 | 176.81 | 0.05804 |
| 15398570 | 109.19 | 0.03584 |
| 9100693  | 108.10 | 0.03548 |
| 28351001 | 106.11 | 0.03483 |
| 30801593 |  65.11 | 0.02137 |
| 2869134  |  51.18 | 0.01680 |
| 9100692  |  46.68 | 0.01532 |
| 29427842 |  43.40 | 0.01425 |

---

## 📂 Outputs

- `../artifacts/bi_exports/campaign_attribution_shapley.csv`  
  Columns: `campaign_id, shapley_credit, share`  
  *(share is normalized over the selected sample; use for BI leaderboards & comparisons)*

- `../artifacts/models/xgb_booster_shapley.json`  
  *(8-feature booster used as the value function `v(S)`)*
  
---

## 🧠 How to Interpret

- **Shapley credit** (per campaign) ≈ *Total marginal lift in modeled conversion probability* attributable to that campaign’s presence across journeys.  
- **Share** = campaign’s portion of total Shapley credit within the evaluated sample.

**Why it’s useful**
- Provides a **data-driven, order-aware** credit signal that complements heuristic rules (First/Last/Linear/Decay).  
- Because contributions are computed by **removing/adding touches**, it aligns with **incrementality intuition**: *What if this touch hadn’t happened?*

---

## ✅ Correctness & Guards

- **Stable list parsing** for sequence columns (arrays/strings/null-safe).  
- **Binary label** is created if missing (from `conversion_id` or `conv_time`).  
- **Global category coding** for first/last ensures consistent integer mappings across all subsets.  
- For `CLICKS_ONLY=True`, the method **falls back** to all views if no clicks exist in a journey.

> Note: Monte Carlo Shapley is an **approximation**; increase `N_PERM` and `N_JOURNEYS` for tighter estimates.

---

## 📈 Tips & Variations

- **Tighten estimates**: try `N_PERM=100–200` and `N_JOURNEYS=20k–50k` (runtime ↑ linearly).  
- **Clicks-only credit**: set `CLICKS_ONLY=True` to emphasize active engagement.  
- **Calibrated value function**: optionally swap in a **calibrated** version of `v(S)` if you want probabilities to be well-calibrated (affects absolute magnitudes, less so rankings).  
- **Consistency checks** (optional): within each permutation, the sum of marginal contributions should equal `v(full_path) − v(empty)`.

---

## 🔌 How this fits the pipeline
1. **Notebook 01 — Data Preprocessing & EDA** 
   Prepare impression-level data, KPIs, and Parquet artifacts.
2. **Notebook 02 — Build User Journeys**  
   Convert impressions → ordered user journeys with sequences.
3. **Notebook 03 — Attribution & Modeling (3a–3c)**  
   - **03a — Heuristic Attribution:** First/Last/Linear/Time-decay over journeys.  
   - **03b — Journey Conversion Modeling:** Logistic Regression & XGBoost with calibration + SHAP.  
   - **03c — Shapley Attribution (Monte Carlo):** Model-based credit per campaign. (This)
4. **Notebook 04 — Attribution Rollups, ROI & Rule-vs-Model Comparison**  
   Consolidate attributions, compute ROI, compare methods.
5. **Notebook 05 — Customer Segmentation & Audience Exports**  
   KMeans segmentation, campaign-mix profiles, and BI exports.
6. **Notebook 06 — Budget Reallocation Simulator (Upcoming)**  
   What-if scenarios using attributed ROI and constraints.

---

## 🛠️ Repro

- Ensure `../data/processed/journeys_all.parquet` exists (from Notebook 02).  
- Run the booster build cell once (saves to `../artifacts/models/xgb_booster_shapley.json`).  
- Adjust `CONV_ONLY`, `N_JOURNEYS`, `N_PERM`, `CLICKS_ONLY` as needed.  
- Final artifact for BI: `../artifacts/bi_exports/campaign_attribution_shapley.csv`.

---


In [1]:
import pandas as pd

# Load processed journeys (already built earlier)
J = pd.read_parquet("../data/processed/journeys_all.parquet")

print("Journeys loaded:", J.shape)
print("Columns:", J.columns.tolist()[:15])


Journeys loaded: (6814951, 13)
Columns: ['uid', 'conversion', 'conversion_id', 'conv_time', 'length', 'campaign_seq', 'click_seq', 'cost_seq', 'time_since_last_click_seq', 'first_campaign_id', 'last_campaign_id', 'total_cost', 'num_clicks']


In [2]:
# Shapley setup: load journeys_all, build 8 engineered features consistently, get a booster
import os, ast, json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss


# Ensure list-like columns are proper Python lists (idempotent)
def to_pylist(x):
    if isinstance(x, list): return x
    if isinstance(x, (np.ndarray, pd.Series)): return x.tolist()
    if isinstance(x, str) and x.strip().startswith('[') and x.strip().endswith(']'):
        try: return ast.literal_eval(x)
        except Exception: return []
    if x is None or (isinstance(x, float) and pd.isna(x)) or x == "": return []
    return [x]

for c in ["campaign_seq","click_seq","cost_seq","time_since_last_click_seq"]:
    if c in J.columns:
        J[c] = J[c].apply(to_pylist)

# Ensure a binary conversion label exists
if "conversion" not in J.columns:
    if "conversion_id" in J.columns:
        J["conversion"] = (J["conversion_id"] != -1).astype(int)
    elif "conv_time" in J.columns:
        J["conversion"] = (J["conv_time"].fillna(-1) >= 0).astype(int)
    else:
        J["conversion"] = 0
print("Has conversion:", "conversion" in J.columns, "| Positives:", int(J["conversion"].sum()))

# Feature builder for the Shapley model (8 features)
def safe_mean_pos(vals):
    vals = [v for v in vals if (isinstance(v,(int,float)) and v >= 0)]
    return float(np.mean(vals)) if len(vals) else 0.0

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    F = pd.DataFrame(index=df.index)
    F["uid"] = df["uid"]
    F["conversion"] = df["conversion"].astype(int)

    # path stats
    F["path_length"] = df["campaign_seq"].apply(len).clip(upper=20)
    F["num_clicks"]  = df["click_seq"].apply(lambda x: int(np.sum(x))).clip(upper=20)
    F["unique_campaigns"] = df["campaign_seq"].apply(lambda x: len(set(x)))

    # cost stats (log1p to stabilize)
    F["total_cost"] = np.log1p(df["cost_seq"].apply(np.sum))
    F["avg_cost"]   = np.log1p(df["cost_seq"].apply(lambda x: float(np.mean(x)) if len(x)>0 else 0.0))

    # time gaps (seconds → log1p of mean over valid (>=0) entries)
    F["avg_time_gap"] = np.log1p(df["time_since_last_click_seq"].apply(safe_mean_pos))

    # first/last campaign IDs as strings
    first_id = df["campaign_seq"].apply(lambda s: str(s[0]) if len(s)>0 else "-1")
    last_id  = df["campaign_seq"].apply(lambda s: str(s[-1]) if len(s)>0 else "-1")

    # build ONE shared category space so codes are consistent across splits/columns
    all_ids = pd.Index(first_id).append(pd.Index(last_id)).astype(str)
    cats = pd.Index(all_ids.unique())

    F["first_campaign"] = pd.Categorical(first_id, categories=cats).codes
    F["last_campaign"]  = pd.Categorical(last_id,  categories=cats).codes
    return F

F = build_features(J)

# Exact feature order used by the Shapley booster 
X_cols = [
    "path_length",
    "num_clicks",
    "total_cost",
    "avg_cost",
    "avg_time_gap",
    "unique_campaigns",
    "first_campaign",
    "last_campaign",
]
print("Features ready:", F[X_cols].shape)

# Load/train a booster on these 8 features ONLY
model_dir = "../artifacts/models"
os.makedirs(model_dir, exist_ok=True)
booster_path = os.path.join(model_dir, "xgb_booster_shapley.json")   # <- separate file for Shapley model

if os.path.exists(booster_path):
    booster = xgb.Booster()
    booster.load_model(booster_path)
    print("Loaded Shapley booster from", booster_path)
else:
    print("No Shapley booster found. Training a small XGB model on the 8 engineered features...")
    X = F[X_cols].values
    y = F["conversion"].astype(int).values

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

    dtr = xgb.DMatrix(X_train, label=y_train)
    dva = xgb.DMatrix(X_val,   label=y_val)
    dte = xgb.DMatrix(X_test,  label=y_test)

    params = dict(
        objective="binary:logistic",
        eval_metric=["logloss","auc"],
        eta=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        seed=42,
    )

    booster = xgb.train(
        params=params,
        dtrain=dtr,
        num_boost_round=1200,
        evals=[(dtr,"train"),(dva,"val")],
        early_stopping_rounds=100,
        verbose_eval=False
    )

    p_val  = booster.predict(dva, iteration_range=(0, booster.best_iteration+1))
    p_test = booster.predict(dte, iteration_range=(0, booster.best_iteration+1))
    print(f"Quick booster AUC (val): {roc_auc_score(y_val, p_val):.3f} | AUC (test): {roc_auc_score(y_test, p_test):.3f}")

    booster.save_model(booster_path)
    print("Saved Shapley booster ->", booster_path)


Has conversion: True | Positives: 806196
Features ready: (6814951, 8)
No Shapley booster found. Training a small XGB model on the 8 engineered features...
Quick booster AUC (val): 0.957 | AUC (test): 0.957
Saved Shapley booster -> ../artifacts/models/xgb_booster_shapley.json


In [5]:
import numpy as np, pandas as pd, os, ast, xgboost as xgb
from tqdm.auto import tqdm

# Load data + booster (the one you just trained on 8 engineered features)

J = pd.read_parquet("../data/processed/journeys_all.parquet")

booster_path = "../artifacts/models/xgb_booster_shapley.json"
booster = xgb.Booster()
booster.load_model(booster_path)

# Keep only columns we need
def to_pylist(x):
    if isinstance(x, list): return x
    if isinstance(x, (np.ndarray, pd.Series)): return x.tolist()
    if isinstance(x, str) and x.strip().startswith("[") and x.strip().endswith("]"):
        try: return ast.literal_eval(x)
        except Exception: return []
    if x is None or (isinstance(x, float) and pd.isna(x)) or x == "": return []
    return [x]

for c in ["campaign_seq","click_seq","cost_seq","time_since_last_click_seq"]:
    if c in J.columns:
        J[c] = J[c].apply(to_pylist)

# Label (if not present)
if "conversion" not in J.columns:
    if "conversion_id" in J.columns:
        J["conversion"] = (J["conversion_id"] != -1).astype(int)
    elif "conv_time" in J.columns:
        J["conversion"] = (J["conv_time"].fillna(-1) >= 0).astype(int)
    else:
        J["conversion"] = 0


# Global category coding used for first/last campaign (CONSISTENT mapping)

first_id_all = J["campaign_seq"].apply(lambda s: str(s[0]) if len(s)>0 else "-1")
last_id_all  = J["campaign_seq"].apply(lambda s: str(s[-1]) if len(s)>0 else "-1")
cats = pd.Index(first_id_all).append(pd.Index(last_id_all)).astype(str).unique()
# ensure "-1" exists
if "-1" not in set(cats): cats = cats.append(pd.Index(["-1"]))

# fast mapping
cat_code_map = {c:i for i, c in enumerate(cats)}
code_for_missing = cat_code_map.get("-1", -1)

# Feature builder for any SUBSET of a single journey
X_cols = [
    "path_length","num_clicks","total_cost","avg_cost",
    "avg_time_gap","unique_campaigns","first_campaign","last_campaign"
]

def safe_mean_pos(vals):
    vals = [v for v in vals if (isinstance(v,(int,float)) and v>=0)]
    return float(np.mean(vals)) if len(vals) else 0.0

def features_from_subset(c_seq, cli_seq, cost_seq, tlc_seq, idx_subset):
    """Build the 8 engineered features from a subset of touchpoint indices."""
    if len(idx_subset) == 0:
        # empty path → all zeros, first/last campaign = -1 code
        return np.array([0,0,0.0,0.0,0.0,0, code_for_missing, code_for_missing], dtype=float)

    # slice lists
    c_sub   = [c_seq[i]   for i in idx_subset]
    cli_sub = [cli_seq[i] for i in idx_subset] if len(cli_seq)>=len(c_seq) else [0]*len(idx_subset)
    cos_sub = [cost_seq[i] for i in idx_subset] if len(cost_seq)>=len(c_seq) else [0.0]*len(idx_subset)
    tlc_sub = [tlc_seq[i] for i in idx_subset] if len(tlc_seq)>=len(c_seq) else []

    path_length     = len(idx_subset)
    num_clicks      = int(np.sum(cli_sub))
    total_cost_raw  = float(np.sum(cos_sub))
    avg_cost_raw    = float(np.mean(cos_sub)) if path_length>0 else 0.0
    avg_tlc_raw     = safe_mean_pos(tlc_sub)

    total_cost      = np.log1p(total_cost_raw)
    avg_cost        = np.log1p(avg_cost_raw)
    avg_time_gap    = np.log1p(avg_tlc_raw)
    unique_campaigns= len(set(c_sub))

    first_id = str(c_sub[0]) if path_length>0 else "-1"
    last_id  = str(c_sub[-1]) if path_length>0 else "-1"
    first_code = cat_code_map.get(first_id, code_for_missing)
    last_code  = cat_code_map.get(last_id,  code_for_missing)

    return np.array([
        path_length, num_clicks, total_cost, avg_cost,
        avg_time_gap, unique_campaigns, first_code, last_code
    ], dtype=float)

def v_prob(x_vector):
    """Model-predicted conversion probability for one feature vector."""
    d = xgb.DMatrix(x_vector.reshape(1, -1))
    return float(booster.predict(d, iteration_range=(0, booster.best_iteration+1)))

# Monte-Carlo Shapley for campaign credit per journey
def shapley_for_journey(c_seq, cli_seq, cost_seq, tlc_seq, n_perm=50, clicks_only=False):
    """Return dict: campaign_id -> Shapley credit for this single journey."""
    n = len(c_seq)
    if n == 0: return {}
    idx_all = [i for i in range(n) if (not clicks_only or (i < len(cli_seq) and int(cli_seq[i])==1))]
    if len(idx_all) == 0:
        idx_all = list(range(n))  # fallback to all views if no clicks

    # Precompute full-set value ; but we need marginal along permutations
    credit = {}
    rng = np.random.default_rng(42)
    for _ in range(n_perm):
        perm = rng.permutation(idx_all)
        S = []
        prev_v = v_prob(features_from_subset(c_seq, cli_seq, cost_seq, tlc_seq, S))
        for i in perm:
            S_plus = S + [i]
            v_new = v_prob(features_from_subset(c_seq, cli_seq, cost_seq, tlc_seq, S_plus))
            delta = v_new - prev_v
            camp = str(c_seq[i])
            credit[camp] = credit.get(camp, 0.0) + delta
            S = S_plus
            prev_v = v_new
    # average over permutations
    for k in credit.keys():
        credit[k] /= float(n_perm)
    return credit

# Run on a sample of CONVERTED journeys and aggregate per campaign

CONV_ONLY = True
N_JOURNEYS = 5000        # increase to 20k/50k later (runtime grows linearly)
N_PERM     = 50          # 100–200 for final runs; 50 is fine for demo
CLICKS_ONLY = False      # set True to credit only clicked touchpoints

J_conv = J[J["conversion"]==1] if CONV_ONLY else J
J_sample = J_conv.sample(n=min(N_JOURNEYS, len(J_conv)), random_state=42)

agg = {}
for r in tqdm(J_sample.itertuples(index=False), total=len(J_sample), desc="Shapley per journey"):
    c_seq   = getattr(r, "campaign_seq", [])
    cli_seq = getattr(r, "click_seq", [])
    cos_seq = getattr(r, "cost_seq", [])
    tlc_seq = getattr(r, "time_since_last_click_seq", [])
    contrib = shapley_for_journey(c_seq, cli_seq, cos_seq, tlc_seq, n_perm=N_PERM, clicks_only=CLICKS_ONLY)
    for camp, val in contrib.items():
        agg[camp] = agg.get(camp, 0.0) + val

camp_df = pd.DataFrame({"campaign_id": list(agg.keys()), "shapley_credit": list(agg.values())})
camp_df = camp_df.sort_values("shapley_credit", ascending=False).reset_index(drop=True)

# Normalize to shares (optional but nice for BI)
total_credit = camp_df["shapley_credit"].sum()
camp_df["share"] = camp_df["shapley_credit"] / total_credit if total_credit > 0 else 0.0

display(camp_df.head(15))

# Save for BI
os.makedirs("../artifacts/bi_exports", exist_ok=True)
out_path = "../artifacts/bi_exports/campaign_attribution_shapley.csv"
camp_df.to_csv(out_path, index=False)
print("Saved ->", out_path,
      f"(journeys={len(J_sample)}, perms/journey={N_PERM}, clicks_only={CLICKS_ONLY})")


Shapley per journey:   0%|          | 0/5000 [00:00<?, ?it/s]

/var/folders/d9/7q11w0fj4h37hzlh3m8w5rx80000gn/T/ipykernel_94771/4150138168.py:97: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(booster.predict(d, iteration_range=(0, booster.best_iteration+1)))


,campaign_id,shapley_credit,share
0,10341182,225.366672,0.073975
1,32368244,191.855976,0.062976
2,15184511,176.808032,0.058036
3,15398570,109.188734,0.035841
4,9100693,108.101168,0.035484
5,28351001,106.112132,0.034831
6,30801593,65.105342,0.021370
7,2869134,51.176978,0.016799
8,9100692,46.681182,0.015323
9,29427842,43.404913,0.014247


Saved -> ../artifacts/bi_exports/campaign_attribution_shapley.csv (journeys=5000, perms/journey=50, clicks_only=False)
